# 노후주택 데이터로 YOLO 학습 + 등급 실증 (Colab, 올인원)

AI-Hub **API 키**만 있으면 다운로드→변환→학습→실증을 **코랩 한 곳에서** 전부 처리합니다. (Mac·드라이브 왕복 불필요)

- 데이터셋: `567` 서울시 노후 주택 균열 데이터
- 시작 대상: **비주거용주택**(=노후 상가류, 우리 타깃) `_231023_add` 세트 → 라벨 `400979`, 이미지 `400984`
- 학습 클래스 3종: crack / spalling / rebar (거주성 항목 대지·마감·생활·창호는 자동 제외)

> **사전 조건**: AI-Hub에서 데이터셋 567 **활용신청 승인 완료** + 그 계정의 **API 키**.

**런타임 → 런타임 유형 변경 → GPU(T4/L4/A100)** 설정 후 위에서부터 실행.

---
⚠️ 만약 3번 다운로드에서 '승인/IP차단' 경고가 뜨면 → 코랩 IP가 막힌 것. 그때만 **폴백**: 내 Mac에서 `download_convert_house.sh`로 받아 `house_yolo.zip`을 드라이브에 올리고 맨 아래 '폴백' 셀 사용.

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. 저장소 clone (변환 스크립트 가져오기)

In [ ]:
!git clone --depth 1 https://github.com/CalainKim/Google_AI_CapstoneDesign_Yolo-Based-Crack-Detection-and-Prediction-System.git /content/repo
%cd /content/repo/data-tools
!ls

## 3. AI-Hub에서 직접 다운로드 (API 키)
실행하면 키 입력창이 뜹니다(화면·노트북에 저장 안 됨). 여러 주택 타입을 추가하려면 `FILES`에 라벨·이미지 filekey 쌍을 더 넣으세요.

In [ ]:
import getpass, os
KEY = getpass.getpass('AI-Hub API key: ').strip()
DATASET = '567'
# {구분: filekey}  — 라벨/이미지 쌍. 타입 추가 예: 아파트 라벨 400980 / 이미지 400985
FILES = {'labels': '400979', 'images': '400984'}
os.makedirs('/content/dl', exist_ok=True)

def download(kind, filekey):
    out = f'/content/dl/{kind}.tar'
    url = f'https://api.aihub.or.kr/down/0.6/{DATASET}.do?fileSn={filekey}'
    print(f'== {kind} (fileSn={filekey}) 다운로드...')
    os.system(f'curl -L -H "apikey:{KEY}" -o "{out}" "{url}"')
    sz = os.path.getsize(out)
    head = open(out, 'rb').read(300)
    blocked = (sz < 10000) or (b'html' in head.lower()) or ('승인'.encode() in head) or ('인증'.encode() in head)
    print(f'   {out}: {sz/1e6:.1f} MB', '⚠️ 실패/차단 의심' if blocked else '✅')
    if blocked:
        print('   응답 앞부분:', head[:200])
        raise SystemExit('다운로드 실패(승인 누락/IP차단 가능). 맨 아래 폴백 방법 사용.')

for kind, fk in FILES.items():
    download(kind, fk)
print('\n다운로드 완료.')

## 4. YOLO 변환 (라벨+이미지 → data.yaml 세트)
폴더 경로의 결함(균열/박리·박락/철근노출)·등급(우수/보통/불량)을 읽어 YOLO 라벨 생성 + `grades.json`(등급 실증용) 산출. (결함×등급) 셀당 최대 800장 균형 샘플.

In [ ]:
# 병합·해제 (구조 보존). 라벨/이미지 각각 별도 트리로 추출.
!python prepare_house_from_tar.py --tar /content/dl/labels.tar --work /content/prep_labels
!python prepare_house_from_tar.py --tar /content/dl/images.tar --work /content/prep_images

# YOLO 변환
!python aihub_house_to_yolo.py \
  --labels-dir /content/prep_labels/extracted \
  --images-dir /content/prep_images/extracted \
  --out-dir /content/house_yolo --defect-from path --max-per-cell 800 --val-ratio 0.2

import glob, os
ROOT = '/content/house_yolo'
yaml_path = os.path.join(ROOT, 'data.yaml')
print('train:', len(glob.glob(f'{ROOT}/images/train/*')), '| val:', len(glob.glob(f'{ROOT}/images/val/*')))
print(open(yaml_path, encoding='utf-8').read())

## 5. YOLO 학습
클래스 3종(crack/spalling/rebar). 더 가볍게: `yolo11n.pt` / 더 정확: `yolo11m.pt`.

In [ ]:
%pip install -q ultralytics
from ultralytics import YOLO

model = YOLO('yolo11s.pt')
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    name='house_crack',
)

## 6. best.pt 저장 (다운로드 + 드라이브 백업)
이 `best.pt` 를 `ai-server/models/best.pt` 에 넣으면 서버가 mock → 실제 추론으로 전환됩니다.

In [ ]:
import os
from IPython.display import Image, display
run_dir = results.save_dir
res_png = os.path.join(run_dir, 'results.png')
if os.path.exists(res_png):
    display(Image(filename=res_png, width=900))
best = os.path.join(run_dir, 'weights', 'best.pt')
print('best.pt:', best)

# 드라이브 백업(선택)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    shutil.copy(best, '/content/drive/MyDrive/house_crack_best.pt')
    print('드라이브 저장: /content/drive/MyDrive/house_crack_best.pt')
except Exception as e:
    print('드라이브 백업 건너뜀:', e)

from google.colab import files
files.download(best)

## 7. ★ 등급 실증 — 우수/보통/불량 vs 우리 A~E (confusion matrix)
교수님이 원하는 **'사용자 체감 최종 정확도'** 실증. 전문가 라벨 없이 데이터 등급 라벨만으로 즉시 검증.

val 이미지에 대해: 탐지 → 간이 트리아지 점수 → A~E → **우수(A·B)/보통(C)/불량(D·E)** 그룹 매핑 → `grades.json`(실제 등급)과 대조. **위험 누락률**(실제 불량을 우수/보통으로 오분류)이 핵심 안전 지표.

In [ ]:
import json, glob, os
from collections import defaultdict

SEVERITY = {'crack': 0.55, 'spalling': 0.85, 'rebar': 0.95}
W_SEV, W_DENS, W_CNT = 0.55, 0.30, 0.15
def score_to_group(score):
    if score >= 60: return '불량'   # D,E
    if score >= 40: return '보통'   # C
    return '우수'                   # A,B

grades = json.load(open(os.path.join(ROOT, 'grades.json'), encoding='utf-8'))
names = model.names
val_imgs = glob.glob(f'{ROOT}/images/val/*')

GROUPS = ['우수', '보통', '불량']
cm = defaultdict(int)
n = 0
for img in val_imgs:
    true_g = grades.get(os.path.basename(img))
    if true_g not in GROUPS:
        continue
    r = model.predict(img, conf=0.25, verbose=False)[0]
    W, H = r.orig_shape[1], r.orig_shape[0]
    area = max(W * H, 1)
    sev = dens = cnt = 0.0
    boxes = r.boxes
    if boxes is not None and len(boxes) > 0:
        for b in boxes:
            cls = names[int(b.cls)]
            sev = max(sev, SEVERITY.get(cls, 0.5))
            x1, y1, x2, y2 = b.xyxy[0].tolist()
            dens += (x2 - x1) * (y2 - y1)
        dens = min(dens / area, 1.0)
        cnt = min(len(boxes) / 10.0, 1.0)
    score = 100 * (W_SEV * sev + W_DENS * dens + W_CNT * cnt)
    pred_g = score_to_group(score)
    cm[(true_g, pred_g)] += 1
    n += 1

print(f'검증 이미지(등급 라벨 有): {n}장\n')
print('실제\\예측 |', ' | '.join(f'{g:>4}' for g in GROUPS))
for t in GROUPS:
    row = [cm[(t, p)] for p in GROUPS]
    print(f'{t:>6}   |', ' | '.join(f'{v:>4}' for v in row))

correct = sum(cm[(g, g)] for g in GROUPS)
acc = correct / n if n else 0
danger_total = sum(cm[('불량', p)] for p in GROUPS)
missed = cm[('불량', '우수')] + cm[('불량', '보통')]
miss_rate = missed / danger_total if danger_total else 0
print(f'\n등급 정확도: {acc:.1%}')
print(f'위험 누락률(실제 불량을 놓침): {miss_rate:.1%}  ← 안전상 가장 중요, 낮을수록 좋음')

## 8. 빠른 추론 시각화 (선택)

In [ ]:
import glob, os
from IPython.display import Image, display
best_model = YOLO(best)
for img in glob.glob(f'{ROOT}/images/val/*')[:3]:
    p = best_model.predict(img, save=True, conf=0.25, verbose=False)[0]
    display(Image(filename=os.path.join(p.save_dir, os.path.basename(p.path)), width=600))

---
## (폴백) 코랩 IP가 막혔을 때만: Mac에서 받은 zip으로 학습
3번에서 차단 경고가 뜬 경우에만. 내 Mac에서 `bash data-tools/download_convert_house.sh 567 "400979,400984" 800` 로 `house_yolo.zip` 생성 → 드라이브 최상위 업로드 후 아래 실행(그 뒤 5번 학습부터 이어서).

In [ ]:
# 폴백 전용 (직접 다운로드가 됐으면 실행하지 마세요)
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, glob
ZIP_PATH = '/content/drive/MyDrive/house_yolo.zip'
assert os.path.exists(ZIP_PATH), f'파일 없음: {ZIP_PATH}'
os.makedirs('/content/house_yolo', exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/house_yolo')
yamls = glob.glob('/content/house_yolo/**/data.yaml', recursive=True)
ROOT = os.path.dirname(yamls[0])
yaml_path = os.path.join(ROOT, 'data.yaml')
lines = open(yaml_path, encoding='utf-8').read().splitlines()
open(yaml_path, 'w', encoding='utf-8').write('\n'.join(f'path: {ROOT}' if ln.startswith('path:') else ln for ln in lines) + '\n')
print('데이터 루트:', ROOT)
print('train:', len(glob.glob(f'{ROOT}/images/train/*')), '| val:', len(glob.glob(f'{ROOT}/images/val/*')))